# ifcfill label encoding for synthetic data

This notebook shows `cat_encoding="label"`, which converts categorical variables into integer codes and stores mappings so `inverse_transform` can decode them later.

## Setup

Install ifcfill with the optional notebook dependencies:

```bash
pip install "ifcfill[examples]"
```

In [ ]:
import pandas as pd

from ifcfill import IFCTransformer

## Create sample data

In [ ]:
df = pd.DataFrame(
    {
        "age": [25, 30, None, 40, 35],
        "salary": [50_000.50, None, 75_000.00, 90_000.25, 62_000.00],
        "city": ["London", None, "Paris", "London", "Amman"],
        "zip_code": ["00123", "00456", None, "00123", "07890"],
        "joined": pd.to_datetime(
            ["2020-01-01", "2021-06-15", None, "2023-03-10", "2022-11-01"]
        ),
        "active": ["yes", "yes", "yes", "yes", "yes"],
    }
)

df

## Fit and transform with label encoding

Encodings are unsupervised and invertible, so transformed data can be used by synthetic data generators and later mapped back to the original table structure.

In [ ]:
transformer = IFCTransformer(
    col_types={"zip_code": "categorical"},
    cat_fill="constant",
    cat_constant="UNKNOWN",
    cat_encoding="label",
)

transformed = transformer.fit_transform(df)
transformed

`city` and `zip_code` are now integer-coded categorical variables.

In [ ]:
transformed.dtypes

## Inspect the category mappings

In [ ]:
transformer.category_mappings_

In [ ]:
transformer.inverse_category_mappings_

## Inverse transform generated-like data

Synthetic generators may return float values for encoded categorical columns. `inverse_transform` rounds and clips those values to the known code range before decoding. If the decoded value is the learned missing category, it becomes a missing value again.

In [ ]:
generated_like = transformed.copy()
generated_like["city"] = [0.1, 1.8, 20.0, -3.0, 1.2]

restored = transformer.inverse_transform(generated_like)
restored